# Parsing an MCSim Model

**Last updated:** 2025-05-01

This notebook examples the parsing step for an MCSim model.

You can load a model from a file or a string, this notebook demonstrates both.

In [1]:
from pathlib import Path
from rich import print

from pymcsimmod.parser import ModelParser

## Loading a model from a file

Make sure we can find the MCSim model:

In [2]:
filename = Path("../../tests/data/pred_prey.model")
assert filename.exists()
print(filename.absolute().resolve())

C:\Users\tjzur\models\PyMCSimMod\tests\data\pred_prey.model

Show the model text:

In [3]:
print(filename.read_text())

#-------------------------------------------------------------------------------
# pred_prey.model
#
# A Lotka-Volterra predator-prey model.
#
# Author: Dustin Kapraun, U.S. EPA, June 2019
#-------------------------------------------------------------------------------


#-------------------------------------------------------------------------------
# STATE VARIABLES for the model (for which ODEs are provided).

States = {
    x,        # Number of rabbits (1000s).
    y,        # Number of foxes (1000s).
};

# End of STATE VARIABLES.
#-------------------------------------------------------------------------------


#-------------------------------------------------------------------------------
# OUTPUT VARIABLES for the model (which can be obtained at any point in time
# as analytic functions of state variables, input variables, and parameters).

Outputs = {};

# End of OUTPUT VARIABLES.
#-------------------------------------------------------------------------------


#-------------------------------------------------------------------------------
# INPUT VARIABLES for the model (which are independent of other variables, and
# which may vary in time).

Inputs = {};

# End of INPUT VARIABLES.
#-------------------------------------------------------------------------------


#-------------------------------------------------------------------------------
# PARAMETERS for the model (which are independent of time).

# Model parameters.
alpha = 0.67;   # Birth rate of rabbits (1/d).
beta = 1.33;    # Death rate of rabbits (1/d per 1000 foxes).
gamma = 1.00;   # Birth rate of foxes (1/d per 1000 rabbits).
delta = 1.00;   # Death rate of foxes (1/d).

# End of PARAMETERS.
#-------------------------------------------------------------------------------


#-------------------------------------------------------------------------------
# MODEL INITIALIZATION section.

Initialize {
    # Assign an initial value for each state variable.
    x = 1.00;     # Initial number of rabbits (1000s).
    y = 0.75;     # Initial number of foxes (1000s).
}

# End of MODEL INITIALIZATION.
#-------------------------------------------------------------------------------


#-------------------------------------------------------------------------------
# DYNAMICS section.

Dynamics {
    # Time rate of change (ODE) for each state variable.
    dt(x) = alpha * x - beta * x * y;
    dt(y) = gamma * x * y - delta * y;
}

# End of DYNAMICS.
#-------------------------------------------------------------------------------


End.

Now, we can parse the file and show the parsed output:

In [4]:
parser = ModelParser()
parsed = parser.parse(filename.read_text())
print(parsed)    

Model(
    sections=[
        StatesSection(type='States', declarations=[Identifier(name='x'), Identifier(name='y')]),
        OutputsSection(type='Outputs', declarations=[]),
        InputsSection(type='Inputs', declarations=[]),
        Statement(lhs=Identifier(name='alpha'), rhs=Number(value=0.67)),
        Statement(lhs=Identifier(name='beta'), rhs=Number(value=1.33)),
        Statement(lhs=Identifier(name='gamma'), rhs=Number(value=1.0)),
        Statement(lhs=Identifier(name='delta'), rhs=Number(value=1.0)),
        InitializeSection(
            type='Initialize',
            statements=[
                Statement(lhs=Identifier(name='x'), rhs=Number(value=1.0)),
                Statement(lhs=Identifier(name='y'), rhs=Number(value=0.75))
            ]
        ),
        DynamicsSection(
            type='Dynamics',
            statements=[
                Statement(
                    lhs=DtVariable(identifier=Identifier(name='x')),
                    rhs=MathematicalExpression(
                        operator='-',
                        lhs=MathematicalExpression(
                            operator='*',
                            lhs=Identifier(name='alpha'),
                            rhs=Identifier(name='x')
                        ),
                        rhs=MathematicalExpression(
                            operator='*',
                            lhs=MathematicalExpression(
                                operator='*',
                                lhs=Identifier(name='beta'),
                                rhs=Identifier(name='x')
                            ),
                            rhs=Identifier(name='y')
                        )
                    )
                ),
                Statement(
                    lhs=DtVariable(identifier=Identifier(name='y')),
                    rhs=MathematicalExpression(
                        operator='-',
                        lhs=MathematicalExpression(
                            operator='*',
                            lhs=MathematicalExpression(
                                operator='*',
                                lhs=Identifier(name='gamma'),
                                rhs=Identifier(name='x')
                            ),
                            rhs=Identifier(name='y')
                        ),
                        rhs=MathematicalExpression(
                            operator='*',
                            lhs=Identifier(name='delta'),
                            rhs=Identifier(name='y')
                        )
                    )
                )
            ]
        )
    ]
)

## Parsing a text model

Alternatively, instead of loading a file from a file, you can load from a string.

In [5]:
text_model = """
States = {
    x,        # Number of rabbits (1000s).
    y,        # Number of foxes (1000s).
};

Outputs = {};
Inputs = {};

alpha = 0.67;   # Birth rate of rabbits (1/d).
beta = 1.33;    # Death rate of rabbits (1/d per 1000 foxes).
gamma = 1.00;   # Birth rate of foxes (1/d per 1000 rabbits).
delta = 1.00;   # Death rate of foxes (1/d).

Initialize {
    # Assign an initial value for each state variable.
    x = 1.00;     # Initial number of rabbits (1000s).
    y = 0.75;     # Initial number of foxes (1000s).
}
Dynamics {
    # Time rate of change (ODE) for each state variable.
    dt(x) = alpha * x - beta * x * y;
    dt(y) = gamma * x * y - delta * y;
}

End.
"""

parser = ModelParser()
parsed_model = parser.parse(text_model)